In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch.utils.data import DataLoader
from torch import nn
from tm_task_generators import contains_one_task_generator
from tm_interface import TMAgent, TMInterface, TMClass, sample_logits
from reviewer_rl import create_population, score_population, score_population_streams, ReviewerDataset, Reviewer, fast_rollout, verbose_rollout, compiled_rollout, score_population_compiled
from reviewer_rl_training import train_reviewer, TrainConfig


In [3]:
tasks = contains_one_task_generator(8, 512)
agent = TMAgent().to('cuda')
interface = TMInterface()
tape, correct_output  = tasks

In [35]:
rewards = verbose_rollout(agent, tasks, interface, max_depth=20)

(2)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: False
(0)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(1)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(1)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(1)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 0 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
 1 |(0)| 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(1)| 1 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 1 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(1)| 1 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(1)| 1 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 1 | 0 | 0 | 0 | 0 | 2 | 2       REWARD: 0.0 HALTED: True
(0)| 1 

In [4]:
train_population = create_population(agent, 1000)

In [15]:
train_scores = score_population_compiled(train_population, tasks, interface)

In [17]:
train_scores = score_population_streams(train_population, tasks, interface)

In [18]:
train_scores = score_population(train_population, tasks, interface)

In [ ]:
train_scores = compiled_rollout(train_population, tasks, interface)

InternalTorchDynamoError: AsPythonConstantNotImplementedError: UnspecializedNNModuleVariable(TMAgent) is not a constant

from user code:
   File "/home/mxmlabserver/Desktop/JPKWork/ReviewerBasedRL/.venv/lib/python3.13/site-packages/torch/_dynamo/external_utils.py", line 69, in inner
    return fn(*args, **kwargs)
  File "/home/mxmlabserver/Desktop/JPKWork/ReviewerBasedRL/.venv/lib/python3.13/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/home/mxmlabserver/Desktop/JPKWork/ReviewerBasedRL/reviewer_rl.py", line 65, in fast_rollout
    logits = agent(initial_agent_state, obs)

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [ ]:
train_dataset = ReviewerDataset(train_population, train_scores)

In [ ]:
test_population = create_population(agent, 200)
test_scores = score_population_streams(test_population, tasks, interface)

In [ ]:
test_dataset = ReviewerDataset(test_population, test_scores)

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=10, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=True,
)

In [ ]:
for p, score in train_loader:
    print(p.shape, score)

torch.Size([10, 971]) tensor([0.5000, 0.4922, 0.5273, 0.4805, 0.5137, 0.5547, 0.4355, 0.5000, 0.4648,
        0.5078], device='cuda:0')
torch.Size([10, 971]) tensor([0.4297, 0.4727, 0.4727, 0.4824, 0.5020, 0.4863, 0.4727, 0.5332, 0.5137,
        0.5352], device='cuda:0')
torch.Size([10, 971]) tensor([0.4707, 0.4805, 0.4883, 0.4766, 0.4980, 0.4707, 0.4668, 0.4668, 0.4688,
        0.4883], device='cuda:0')
torch.Size([10, 971]) tensor([0.5352, 0.5078, 0.5137, 0.5059, 0.4980, 0.4688, 0.4355, 0.5508, 0.5176,
        0.4922], device='cuda:0')
torch.Size([10, 971]) tensor([0.5195, 0.4902, 0.4707, 0.5000, 0.4531, 0.5293, 0.4863, 0.4902, 0.5039,
        0.4824], device='cuda:0')
torch.Size([10, 971]) tensor([0.4883, 0.4727, 0.5059, 0.4531, 0.5234, 0.4590, 0.4863, 0.5156, 0.5176,
        0.4473], device='cuda:0')
torch.Size([10, 971]) tensor([0.4883, 0.4727, 0.4473, 0.5312, 0.4590, 0.4922, 0.4805, 0.5293, 0.4688,
        0.5156], device='cuda:0')
torch.Size([10, 971]) tensor([0.4668, 0.4453, 0.

In [ ]:
reviewer = Reviewer(agent.parameters_vector()).to('cuda')

In [ ]:
reviewer(test_population[2].parameters_vector().to('cuda'))

tensor([0.5084], device='cuda:0', grad_fn=<SigmoidBackward0>)

In [ ]:
train_config = TrainConfig(epochs=200, early_stopping_patience=15)
print(train_config)

TrainConfig(epochs=200, batch_size=128, lr=0.0003, weight_decay=0.01, warmup_epochs=5, grad_clip_norm=1.0, val_fraction=0.1, early_stopping_patience=15, use_amp=True, num_workers=0, device=None, log_every=1)


In [ ]:
results = train_reviewer(reviewer, train_dataset, test_dataset, config=train_config)

epoch    1/200 | train_loss 0.0007 | val_loss 0.0007 | lr 6.00e-05
epoch    2/200 | train_loss 0.0006 | val_loss 0.0006 | lr 1.20e-04
epoch    3/200 | train_loss 0.0005 | val_loss 0.0006 | lr 1.80e-04
epoch    4/200 | train_loss 0.0005 | val_loss 0.0006 | lr 2.40e-04
epoch    5/200 | train_loss 0.0005 | val_loss 0.0006 | lr 3.00e-04
epoch    6/200 | train_loss 0.0005 | val_loss 0.0005 | lr 3.00e-04
epoch    7/200 | train_loss 0.0005 | val_loss 0.0005 | lr 3.00e-04
epoch    8/200 | train_loss 0.0005 | val_loss 0.0006 | lr 3.00e-04
epoch    9/200 | train_loss 0.0005 | val_loss 0.0005 | lr 3.00e-04
epoch   10/200 | train_loss 0.0005 | val_loss 0.0006 | lr 3.00e-04
epoch   11/200 | train_loss 0.0005 | val_loss 0.0005 | lr 2.99e-04
epoch   12/200 | train_loss 0.0005 | val_loss 0.0005 | lr 2.99e-04
epoch   13/200 | train_loss 0.0005 | val_loss 0.0006 | lr 2.99e-04
epoch   14/200 | train_loss 0.0005 | val_loss 0.0005 | lr 2.98e-04
epoch   15/200 | train_loss 0.0005 | val_loss 0.0005 | lr 2.98

10000 samples: val_loss 0.0013

In [ ]:
train_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=0, pin_memory=False, drop_last=True,
)

criterion = torch.nn.MSELoss()

running_loss = 0.0
for p, score in train_loader:
    pred = reviewer(p)
    loss = criterion(pred, score)
    running_loss += loss
    print("True Score:", float(score.item()), "Pred Score:", float(pred.item()), "Loss:", float(loss.item()))
mean_loss = running_loss/len(train_loader)
print(f"Mean Loss: {mean_loss:.6f}")

True Score: 0.47265625 Pred Score: 0.47500574588775635 Loss: 5.520130798686296e-06
True Score: 0.46875 Pred Score: 0.49634331464767456 Loss: 0.0007613910129293799
True Score: 0.4609375 Pred Score: 0.45467618107795715 Loss: 3.9204114727908745e-05
True Score: 0.49609375 Pred Score: 0.4772011339664459 Loss: 0.0003569309483282268
True Score: 0.4453125 Pred Score: 0.4822600483894348 Loss: 0.0013651213375851512
True Score: 0.4921875 Pred Score: 0.47097039222717285 Loss: 0.00045016565127298236
True Score: 0.4765625 Pred Score: 0.495893269777298 Loss: 0.00037367866025306284
True Score: 0.505859375 Pred Score: 0.47985509037971497 Loss: 0.0006762228440493345
True Score: 0.470703125 Pred Score: 0.4860798716545105 Loss: 0.0002364443353144452
True Score: 0.4453125 Pred Score: 0.46526169776916504 Loss: 0.00039797049248591065
True Score: 0.46484375 Pred Score: 0.47713345289230347 Loss: 0.00015103680198080838
True Score: 0.52734375 Pred Score: 0.4789215624332428 Loss: 0.002344708191230893
True Score: 